## OPENCV DAY 2 : Face Detection and Recognition

In [1]:
import cv2 as cv
import numpy as np
import os

In [2]:
img = cv.imread('../../assests/interview.png')
cv.imshow('Image', img)

gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
cv.imshow('Gray Person', gray)

haar_cascade = cv.CascadeClassifier('haar_face.xml')
faces_rect = haar_cascade.detectMultiScale(gray, scaleFactor = 1.1, minNeighbors = 3)
print(f'Number of faces found = {len(faces_rect)}')

for(x,y,w,h) in faces_rect :
    cv.rectangle(img, (x,y), (x+w,y+h), (0,255,0), thickness = 2)

cv.imshow('Detected Faces', img)
cv.waitKey(0)

Number of faces found = 1


-1

In [3]:
img2 = cv.imread('../../assests/friends.jpeg')
cv.imshow('Image', img2)

gray = cv.cvtColor(img2, cv.COLOR_BGR2GRAY)
cv.imshow('Gray Person', gray)

haar_cascade = cv.CascadeClassifier('haar_face.xml')
faces_rect = haar_cascade.detectMultiScale(gray, scaleFactor = 1.1, minNeighbors = 3)
print(f'Number of faces found = {len(faces_rect)}')

for(x,y,w,h) in faces_rect :
    cv.rectangle(img2, (x,y), (x+w,y+h), (0,255,0), thickness = 2)

cv.imshow('Detected Faces', img2)
cv.waitKey(0)

Number of faces found = 6


-1

In [4]:
import cv2 as cv
haar_cascade = cv.CascadeClassifier('haar_face.xml')

capture = cv.VideoCapture(0)

if not capture.isOpened():
    print("Error: Could not open webcam. Please check if the camera is connected and not in use.")
    exit()

print("Press 'd' to exit the live detection.")

while True:
    isTrue, frame = capture.read()

    if not isTrue:
        print("Failed to grab frame.")
        break

    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    faces_rect = haar_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3)

    print(f'Number of faces found = {len(faces_rect)}') 
    
    for (x, y, w, h) in faces_rect:
        cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), thickness=2)

    cv.imshow('Live Face Detection', frame)

    if cv.waitKey(1) & 0xFF == ord('d'): 
        break

capture.release()
cv.destroyAllWindows()
print("Webcam feed stopped.")

Press 'd' to exit the live detection.
Number of faces found = 1
Number of faces found = 2
Number of faces found = 2
Number of faces found = 1
Number of faces found = 2
Number of faces found = 1
Number of faces found = 2
Number of faces found = 2
Number of faces found = 2
Number of faces found = 2
Number of faces found = 1
Number of faces found = 2
Number of faces found = 2
Number of faces found = 2
Number of faces found = 2
Number of faces found = 2
Number of faces found = 2
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1
Number of faces found = 1


#### Imports and Configuration

In [5]:
import cv2 as cv
import numpy as np
import os

BASE_DIR = r'C:\Users\aryad\Desktop\NeuraMonks-Internship\notebooks\day 31'
TRAIN_DIR = os.path.join(BASE_DIR, 'Faces', 'train')
TEST_DIR = os.path.join(BASE_DIR, 'Faces', 'test')
HAAR_CASCADE_PATH = os.path.join(BASE_DIR, 'haar_face.xml')

PEOPLE = ['Chandler', 'Joey', 'Monica', 'Phoebe', 'Rachel', 'Ross']

MODEL_FILE = 'friends_face_trained.yml'
FEATURES_FILE = 'features.npy'
LABELS_FILE = 'labels.npy'

#### Utility Functions

In [6]:
def load_haar_cascade(path):
    cascade = cv.CascadeClassifier(path)
    if cascade.empty():
        print(f"Error: Could not load Haar cascade classifier from {path}")
        exit()
    return cascade

def load_image(image_path):
    img_array = cv.imread(image_path)
    if img_array is None:
        print(f"Warning: Could not load image from {image_path}. Skipping.")
        return None
    return img_array

#### Training Function

In [7]:
def train_face_recognizer(train_data_dir, people_list, haar_cascade_classifier):
    features = []
    labels = []
    print(f"Starting training from: {train_data_dir}")

    for person_index, person_name in enumerate(people_list):
        path = os.path.join(train_data_dir, person_name)
        label = person_index
        print(f"  Processing '{person_name}' (Label: {label}) from {path}")

        if not os.path.isdir(path):
            print(f"Warning: Directory {path} for '{person_name}' does not exist. Skipping.")
            continue

        for img_name in os.listdir(path):
            img_path = os.path.join(path, img_name)
            img_array = load_image(img_path)

            if img_array is None:
                continue

            gray = cv.cvtColor(img_array, cv.COLOR_BGR2GRAY)
            faces_rect = haar_cascade_classifier.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3)

            for (x, y, w, h) in faces_rect:
                faces_roi = gray[y:y+h, x:x+w]
                features.append(faces_roi)
                labels.append(label)

    print('========== Training data collection done ==========')

    if not features:
        print("Error: No features collected. Training cannot proceed.")
        return None, None, None

    features_np = np.array(features, dtype='object')
    labels_np = np.array(labels)

    face_recognizer = cv.face.LBPHFaceRecognizer_create()
    face_recognizer.train(features_np, labels_np)

    print('========== Face Recognizer training complete ==========')
    print(f'Length of collected features = {len(features_np)}')
    print(f'Length of collected labels = {len(labels_np)}')

    return face_recognizer, features_np, labels_np

#### Prediction Function

In [8]:
def recognize_face(image_path, face_recognizer, haar_cascade_classifier, people_list):
    img = load_image(image_path)
    if img is None:
        return

    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    faces_rect = haar_cascade_classifier.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3)

    # Corrected check: Check if the array is empty using .size or .shape
    if faces_rect.size == 0: # This is the most robust way to check if the array is empty
        print(f"No faces detected in {image_path}")
        cv.imshow('Detected Face', img)
    else:
        for (x, y, w, h) in faces_rect:
            face_roi = gray[y:y+h, x:x+w]

            label, confidence = face_recognizer.predict(face_roi)
            predicted_person = people_list[label]
            print(f'Detected: {predicted_person} with confidence of {confidence:.2f}')

            text = f"{predicted_person} ({confidence:.2f})"
            cv.putText(img, text, (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), thickness=2)
            cv.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), thickness=2)

        cv.imshow('Detected Face', img)

    cv.waitKey(0)
    cv.destroyAllWindows()

#### Main Execution Block

In [9]:
haar_cascade = load_haar_cascade(HAAR_CASCADE_PATH)

if os.path.exists(MODEL_FILE) and os.path.exists(FEATURES_FILE) and os.path.exists(LABELS_FILE):
    print("Loading existing trained model and data...")
    face_recognizer = cv.face.LBPHFaceRecognizer_create()
    face_recognizer.read(MODEL_FILE)
    features_loaded = np.load(FEATURES_FILE, allow_pickle=True)
    labels_loaded = np.load(LABELS_FILE)
    print("Model and data loaded successfully.")
else:
    print("No existing model found. Starting training process...")
    face_recognizer, features, labels = train_face_recognizer(TRAIN_DIR, PEOPLE, haar_cascade)

    if face_recognizer is None:
        print("Training failed. Exiting.")
        exit()

    face_recognizer.save(MODEL_FILE)
    np.save(FEATURES_FILE, features)
    np.save(LABELS_FILE, labels)
    print("Trained model and data saved.")

print("\n--- Starting Face Recognition on Test Image ---")
test_image_path = os.path.join(TEST_DIR, 'ross_14.jpeg')
recognize_face(test_image_path, face_recognizer, haar_cascade, PEOPLE)

Loading existing trained model and data...
Model and data loaded successfully.

--- Starting Face Recognition on Test Image ---
Detected: Ross with confidence of 86.83
